# Transformer Foundations, Part 8: Pretrain a Base Model

## Start With the Job: Turn Random Weights Into a Next-Token Predictor

The architecture from Part 6 can move information, but random weights have not learned which movements help language prediction. The shards from Part 7 provide rows of next-token lessons. Pretraining connects them with one repeated loop:

```mermaid
flowchart LR
    A["Packed token IDs"] --> B["Embeddings + position"]
    B --> C["Attention: gather context"]
    C --> D["FFN: refine each token"]
    D --> E["Vocabulary logits"]
    E --> F["Compare with actual next tokens"]
    F --> G["One scalar loss"]
    G --> H["Backpropagate responsibility"]
    H --> I["Optimizer updates learned weights"]
    I --> B
```

Backpropagation is not a separate model stage used during generation. It is the training-time credit-assignment process that asks: **which embeddings, attention projections, FFN gates, normalization scales, and output weights contributed to this error, and in which direction should each move?**

One packed block supplies many lessons in parallel:

```text
visible prefix              target
aria                         heard
aria heard                   the
aria heard the               signal
```

The causal mask prevents each position from reading its own answer. Cross-entropy rewards probability placed on the actual next token. One backward pass then produces gradients for every trainable component that helped create those logits.

## 0. The Challenge

> **The mission:** Turn one randomly initialized modern decoder into a validated, reloadable base-model checkpoint using only the frozen local corpus artifacts.

**What we know so far:**
- Part 7 defines a tokenizer, separate train and validation shards, and their SHA-256 identities.
- The decoder contract fixes RMSNorm, RoPE, grouped-query attention, SwiGLU, and tied embeddings.
- **But random weights have not practiced even one real next-token transition.**

**What this chapter unlocks:** a complete, measured learning loop with trusted inputs, finite-state checks, gradient coverage, controlled optimizer updates, held-out validation, faithful resume, and final reload parity. The result is a small base model, not an assistant.

### Roadmap

| Part | Visible question | Evidence you will produce |
|---|---|---|
| 1 | Can these local artifacts be trusted? | Schema and digest checks |
| 2 | How do logits become one learning signal? | Shifted targets and per-position loss |
| 3 | What keeps updates controlled? | Learning rate, clipping, and accumulation measurements |
| 4 | What changes during training? | Real loss, gradient, and prediction snapshots |
| 5 | Can the run resume exactly enough to continue? | One-update restored-state parity |
| 6 | Which checkpoint should survive? | Validation-only selection |
| 7 | Which components actually moved? | Diffs for embeddings, attention, FFNs, norms, and head |
| 8 | What changes at larger scale? | Labeled parameter and memory estimates |
| 9 | Is the final package complete? | Clean reload logits and generation parity |

**Predict:** If the final loss falls but attention parameters never receive gradients, did the Transformer learn correctly? Part 2 checks gradient coverage instead of trusting the scalar alone.

In [ ]:
# ── Reproducible Local Setup ───────────────────────────────────────────────
from __future__ import annotations

import copy
import hashlib
import json
import math
import os
import platform
import random
import shutil
import time
import tracemalloc
from dataclasses import asdict, dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import HTML, display
from matplotlib.animation import FuncAnimation
from tokenizers import Tokenizer

SEED = 2606
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
QUICK_RUN = True  # CHANGE THIS: set False only for an intentional larger run.
ARTIFACT_ROOT = Path("artifacts/base-lm")
CHECKPOINT_ROOT = ARTIFACT_ROOT / "checkpoints"
FINAL_ROOT = ARTIFACT_ROOT / "final"
NO_PRETRAINED_WEIGHTS_LOADED = True
PLOT_BG = "#1a1a2e"
COLORS = ["#60a5fa", "#f59e0b", "#22c55e", "#ef4444", "#a78bfa"]

print(f"Device: {DEVICE}; QUICK_RUN={QUICK_RUN}; seed={SEED}")
print("PASS: this notebook contains no pretrained-weight loading path")

## 1 · Refuse Untrusted Inputs Before Training

A shard can still parse after one byte changes. That is the dangerous failure: training proceeds on data different from the manifest. Riverside checks identity before allocating the model.

```mermaid
flowchart LR
    A["dataset manifest"] --> B{"Required fields?"}
    C["train + validation shards"] --> D{"SHA-256 matches?"}
    E["model config"] --> F{"Valid modern config?"}
    B --> G["Construct model"]
    D --> G
    F --> G
    B --> H["Refuse with guidance"]
    D --> H
    F --> H
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style H fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Warning:** If the dataset manifest, either shard, or `tokenizer/tokenizer.json` is absent, stop and execute Notebook 07. Do not invent replacement data here. If only `model-config.json` is absent, this notebook writes and announces the CPU-safe fallback: 2 layers, width 64, 4 query heads, 2 key/value heads, SwiGLU width 176, and the dataset's context/vocabulary contract.

**Predict:** A shard has the right byte length but the wrong digest. Should the notebook warn and continue, repair the manifest, or refuse? The next cell resolves it.

In [ ]:
# ── Frozen Artifact Contract ───────────────────────────────────────────────
DATASET_FIELDS = {
    "schema_version", "seed", "tokenizer_type", "vocab_size", "eod_token",
    "context_length", "dtype", "source_documents", "train_documents",
    "validation_documents", "excluded_documents", "exact_duplicate_groups",
    "near_duplicate_groups", "train_tokens", "validation_tokens",
    "train_sha256", "validation_sha256",
}
MODEL_FIELDS = {
    "architecture", "vocab_size", "context_length", "d_model", "n_layers",
    "n_query_heads", "n_kv_heads", "d_ff", "rope_base", "rms_norm_eps",
    "tie_embeddings",
}

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def sha256_json(payload: dict) -> str:
    canonical = json.dumps(payload, sort_keys=True, separators=(",", ":")).encode()
    return hashlib.sha256(canonical).hexdigest()

def require_fields(payload: dict, required: set[str], label: str) -> None:
    missing = sorted(required - payload.keys())
    if missing:
        raise ValueError(f"{label} is missing required fields: {missing}")

def default_model_config(dataset_manifest: dict) -> dict:
    return {
        "architecture": "tiny_modern_decoder",
        "vocab_size": int(dataset_manifest["vocab_size"]),
        "context_length": int(dataset_manifest["context_length"]),
        "d_model": 64, "n_layers": 2, "n_query_heads": 4,
        "n_kv_heads": 2, "d_ff": 176, "rope_base": 10000.0,
        "rms_norm_eps": 1e-5, "tie_embeddings": True,
    }

def load_and_validate_contracts(root: Path) -> tuple[dict, dict, Tokenizer, dict]:
    manifest_path = root / "dataset-manifest.json"
    train_path, validation_path = root / "train.bin", root / "validation.bin"
    tokenizer_path = root / "tokenizer" / "tokenizer.json"
    missing = [str(path) for path in [manifest_path, train_path, validation_path, tokenizer_path] if not path.exists()]
    if missing:
        raise FileNotFoundError(
            "Notebook 07 artifacts are required. Execute 07-pretraining-data-pipeline.ipynb "
            f"from a fresh kernel, then rerun this cell. Missing: {missing}"
        )
    dataset_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    require_fields(dataset_manifest, DATASET_FIELDS, "dataset manifest")
    if dataset_manifest["dtype"] != "uint16":
        raise ValueError("Refusing shard: frozen dtype contract requires uint16")
    actual_hashes = {"train": sha256_file(train_path), "validation": sha256_file(validation_path)}
    expected_hashes = {"train": dataset_manifest["train_sha256"], "validation": dataset_manifest["validation_sha256"]}
    if actual_hashes != expected_hashes:
        raise ValueError(f"Digest refusal: expected {expected_hashes}, measured {actual_hashes}")
    if train_path.resolve() == validation_path.resolve() or actual_hashes["train"] == actual_hashes["validation"]:
        raise ValueError("Training and validation must be distinct shards with distinct content")
    config_path = root / "model-config.json"
    if config_path.exists():
        model_config = json.loads(config_path.read_text(encoding="utf-8"))
        print("Loaded model-config.json from Notebook 06")
    else:
        model_config = default_model_config(dataset_manifest)
        root.mkdir(parents=True, exist_ok=True)
        config_path.write_text(json.dumps(model_config, indent=2) + "\n", encoding="utf-8")
        print("FALLBACK: model-config.json was absent; wrote the documented CPU-safe default")
    require_fields(model_config, MODEL_FIELDS, "model config")
    model_config["vocab_size"] = int(dataset_manifest["vocab_size"])
    if model_config["architecture"] != "tiny_modern_decoder":
        raise ValueError("Refusing config: architecture must be tiny_modern_decoder")
    if model_config["d_model"] % model_config["n_query_heads"] or model_config["n_query_heads"] % model_config["n_kv_heads"]:
        raise ValueError("Refusing config: query heads must divide width and group evenly over KV heads")
    optional_digest = dataset_manifest.get("model_config_sha256")
    config_digest = sha256_json(model_config)
    if optional_digest and optional_digest != config_digest:
        raise ValueError("Config digest does not match the digest recorded by the dataset manifest")
    tokenizer = Tokenizer.from_file(str(tokenizer_path))
    if tokenizer.get_vocab_size() != model_config["vocab_size"]:
        raise ValueError("Tokenizer vocabulary size disagrees with the frozen manifests")
    identities = {
        "config_digest": config_digest,
        "dataset_manifest_digest": sha256_file(manifest_path),
        "tokenizer_digest": sha256_file(tokenizer_path),
        "train_sha256": actual_hashes["train"], "validation_sha256": actual_hashes["validation"],
    }
    print("PASS: schema, dtype, tokenizer, distinct shards, and both shard digests verified")
    return dataset_manifest, model_config, tokenizer, identities

dataset_manifest, model_config, tokenizer, identities = load_and_validate_contracts(ARTIFACT_ROOT)

**Code Walkthrough**

The validator fails closed. It checks fields before values, hashes files from disk, rejects identical train/validation content, and only then loads the local tokenizer. The model config fallback is visible and deterministic; no network API or pretrained checkpoint appears anywhere.

**Checkpoint:** The input boundary moved from "files exist" to "schema, identity, vocabulary, and split are verified."

**Your turn:** Make a temporary copy of a shard outside this repository, flip one byte, and point `load_and_validate_contracts` at that temporary root. The expected result is `Digest refusal`, not training.

## 2 · Random TinyModernLM: Same Architecture, No Learned Corpus Yet

The architecture can be correct while its predictions remain untrained. You will assemble the compact modern decoder locally, then measure finite logits, tied storage, causal behavior, and gradient reach before calling it trainable.

```mermaid
flowchart LR
    A["Token IDs"] --> B["Tied embedding"]
    B --> C["RMSNorm + RoPE GQA"]
    C --> D["RMSNorm + SwiGLU"]
    D --> E["Repeated blocks"]
    E --> F["Final RMSNorm"]
    F --> G["Tied LM head"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Predict:** Before training, will the highest-probability next token be evidence of knowledge, or simply one deterministic consequence of the seed? The measured distribution resolves this without anthropomorphizing the model.

In [ ]:
# ── Compact TinyModernLM ───────────────────────────────────────────────────
class RMSNorm(nn.Module):
    def __init__(self, width: int, eps: float):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(width))
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        scale = x.float().pow(2).mean(dim=-1, keepdim=True).add(self.eps).rsqrt().to(x.dtype)
        return x * scale * self.weight

def build_rope(length: int, head_dim: int, base: float, device: torch.device):
    frequencies = 1.0 / (base ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))
    angles = torch.outer(torch.arange(length, device=device).float(), frequencies)
    return angles.cos()[None, None], angles.sin()[None, None]

def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    even, odd = x[..., 0::2], x[..., 1::2]
    return torch.stack((even * cos - odd * sin, even * sin + odd * cos), dim=-1).flatten(-2)

class GroupedQueryAttention(nn.Module):
    def __init__(self, config: dict):
        super().__init__()
        self.n_query_heads = config["n_query_heads"]
        self.n_kv_heads = config["n_kv_heads"]
        self.head_dim = config["d_model"] // self.n_query_heads
        self.groups = self.n_query_heads // self.n_kv_heads
        self.rope_base = config["rope_base"]
        self.q_proj = nn.Linear(config["d_model"], self.n_query_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(config["d_model"], self.n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(config["d_model"], self.n_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(config["d_model"], config["d_model"], bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch, length, _ = x.shape
        q = self.q_proj(x).view(batch, length, self.n_query_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(batch, length, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(batch, length, self.n_kv_heads, self.head_dim).transpose(1, 2)
        cos, sin = build_rope(length, self.head_dim, self.rope_base, x.device)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        k = k.repeat_interleave(self.groups, dim=1)
        v = v.repeat_interleave(self.groups, dim=1)
        attended = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.o_proj(attended.transpose(1, 2).contiguous().view(batch, length, -1))

class SwiGLU(nn.Module):
    def __init__(self, config: dict):
        super().__init__()
        self.gate = nn.Linear(config["d_model"], config["d_ff"], bias=False)
        self.up = nn.Linear(config["d_model"], config["d_ff"], bias=False)
        self.down = nn.Linear(config["d_ff"], config["d_model"], bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.down(F.silu(self.gate(x)) * self.up(x))

class ModernDecoderBlock(nn.Module):
    def __init__(self, config: dict):
        super().__init__()
        self.attn_norm = RMSNorm(config["d_model"], config["rms_norm_eps"])
        self.attention = GroupedQueryAttention(config)
        self.ffn_norm = RMSNorm(config["d_model"], config["rms_norm_eps"])
        self.ffn = SwiGLU(config)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attention(self.attn_norm(x))
        return x + self.ffn(self.ffn_norm(x))

class TinyModernLM(nn.Module):
    def __init__(self, config: dict):
        super().__init__()
        require_fields(config, MODEL_FIELDS, "TinyModernLM config")
        self.config = dict(config)
        self.token_embedding = nn.Embedding(config["vocab_size"], config["d_model"])
        self.blocks = nn.ModuleList([ModernDecoderBlock(config) for _ in range(config["n_layers"])])
        self.final_norm = RMSNorm(config["d_model"], config["rms_norm_eps"])
        self.lm_head = nn.Linear(config["d_model"], config["vocab_size"], bias=False)
        if config["tie_embeddings"]:
            self.lm_head.weight = self.token_embedding.weight
        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(module: nn.Module) -> None:
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, token_ids: torch.Tensor, targets: torch.Tensor | None = None):
        if token_ids.shape[1] > self.config["context_length"]:
            raise ValueError("Input exceeds configured context length")
        hidden = self.token_embedding(token_ids)
        for block in self.blocks:
            hidden = block(hidden)
        logits = self.lm_head(self.final_norm(hidden))
        loss = None if targets is None else F.cross_entropy(logits.flatten(0, 1), targets.flatten())
        return logits, loss

    @torch.no_grad()
    def generate(self, prompt_ids: list[int], max_new_tokens: int) -> list[int]:
        self.eval()
        generated = list(prompt_ids)
        for _ in range(max_new_tokens):
            window = generated[-self.config["context_length"]:]
            token_ids = torch.tensor([window], dtype=torch.long, device=next(self.parameters()).device)
            logits, _ = self(token_ids)
            generated.append(int(logits[0, -1].argmax()))
        return generated

def decode_ids(token_ids: list[int]) -> str:
    return tokenizer.decode(token_ids, skip_special_tokens=False)

def assert_finite_tensor(name: str, tensor: torch.Tensor) -> None:
    if not torch.isfinite(tensor).all():
        raise FloatingPointError(f"NaN/Inf detected in {name}")

**Code Walkthrough**

Query heads keep separate questions while each group repeats one key/value pair. RoPE changes query/key orientation inside attention; SwiGLU gates candidate features; RMSNorm controls vector scale without recentering. The embedding matrix and output head share one parameter. Generation remains a plain greedy next-token loop: no chat template, retrieval layer, or serving API is hidden inside it.

In [ ]:
# ── Random-Initialization Health Check ─────────────────────────────────────
model = TinyModernLM(model_config).to(DEVICE)
initial_state = {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}
parameter_count = sum(parameter.numel() for parameter in model.parameters())
assert NO_PRETRAINED_WEIGHTS_LOADED
assert model.token_embedding.weight.data_ptr() == model.lm_head.weight.data_ptr()

probe_text = "Aria heard the signal"
probe_ids = tokenizer.encode(probe_text).ids
if not probe_ids:
    raise ValueError("Probe prompt encoded to zero tokens")
probe_ids = probe_ids[-model_config["context_length"]:]
probe_tensor = torch.tensor([probe_ids], dtype=torch.long, device=DEVICE)
initial_logits, _ = model(probe_tensor)
assert_finite_tensor("initial logits", initial_logits)
initial_distribution = initial_logits[0, -1].softmax(-1).detach().cpu()
initial_generation_ids = model.generate(probe_ids, max_new_tokens=12)
initial_generation = decode_ids(initial_generation_ids)

model.zero_grad(set_to_none=True)
smoke_targets = torch.roll(probe_tensor, shifts=-1, dims=1)
_, smoke_loss = model(probe_tensor, smoke_targets)
assert smoke_loss is not None
assert_finite_tensor("initial loss", smoke_loss)
smoke_loss.backward()
expected_surfaces = [
    "token_embedding", "attention.q_proj", "attention.k_proj", "attention.v_proj",
    "attention.o_proj", "ffn.gate", "ffn.up", "ffn.down", "final_norm",
]
for surface in expected_surfaces:
    gradients = [parameter.grad for name, parameter in model.named_parameters() if surface in name]
    if not gradients or any(gradient is None or not torch.isfinite(gradient).all() for gradient in gradients):
        raise AssertionError(f"Missing or non-finite gradient at expected surface: {surface}")
model.zero_grad(set_to_none=True)

print(f"Parameters: {parameter_count:,}; initial probe loss: {smoke_loss.item():.4f}")
print(f"Before-training greedy continuation: {initial_generation!r}")
print("PASS: finite logits/loss, tied weights, and gradients across every expected component")
print("Interpretation: the continuation is seeded random behavior, not learned knowledge")

## 3 · One Packed Block Becomes Many Next-Token Lessons

A packed shard is one token stream. Training uses adjacent windows: the input stops one token earlier, and the target starts one token later. Every position asks the same concrete question: what came next here?

```mermaid
flowchart LR
    A["Packed token stream"] --> B["Input t0 through tN-1"]
    A --> C["Targets t1 through tN"]
    B --> D["TinyModernLM logits"]
    C --> E["Per-position loss"]
    D --> E
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

This reuses the causal-loss intuition from Notebook 04. Same logic, now operating on the frozen Riverside stream.

**Predict:** Does one block produce one lesson or `sequence_length` lessons? The loss vector below makes the count visible.

In [ ]:
# ── Packed Shards and Shifted Targets ──────────────────────────────────────
class PackedShard:
    def __init__(self, path: Path, expected_tokens: int):
        self.path = path
        self.tokens = np.memmap(path, dtype=np.uint16, mode="r")
        if len(self.tokens) != int(expected_tokens):
            raise ValueError(f"{path.name} token count disagrees with the manifest")

    def sample(self, batch_size: int, sequence_length: int, generator: torch.Generator):
        max_start = len(self.tokens) - sequence_length - 1
        if max_start < 1:
            raise ValueError(f"{self.path.name} is too short for sequence_length={sequence_length}")
        starts = torch.randint(0, max_start + 1, (batch_size,), generator=generator).tolist()
        windows = np.stack([
            np.asarray(self.tokens[start:start + sequence_length + 1], dtype=np.int64)
            for start in starts
        ])
        batch = torch.from_numpy(windows).long().to(DEVICE)
        return batch[:, :-1], batch[:, 1:]

    def fixed(self, batch_size: int, sequence_length: int):
        max_start = len(self.tokens) - sequence_length - 1
        starts = np.linspace(0, max_start, batch_size, dtype=int)
        windows = np.stack([
            np.asarray(self.tokens[start:start + sequence_length + 1], dtype=np.int64)
            for start in starts
        ])
        batch = torch.from_numpy(windows).long().to(DEVICE)
        return batch[:, :-1], batch[:, 1:]

train_shard = PackedShard(ARTIFACT_ROOT / "train.bin", dataset_manifest["train_tokens"])
validation_shard = PackedShard(ARTIFACT_ROOT / "validation.bin", dataset_manifest["validation_tokens"])
assert train_shard.path.resolve() != validation_shard.path.resolve()
sequence_length = min(model_config["context_length"], 64 if QUICK_RUN else model_config["context_length"])

preview_x, preview_y = train_shard.sample(1, sequence_length, torch.Generator().manual_seed(SEED + 1))
with torch.no_grad():
    preview_logits, _ = model(preview_x)
    preview_losses = F.cross_entropy(preview_logits[0], preview_y[0], reduction="none")
assert preview_x.shape == preview_y.shape == (1, sequence_length)
assert torch.equal(preview_x[:, 1:], preview_y[:, :-1])
assert_finite_tensor("per-position loss", preview_losses)
print("Decoded packed training window:")
print(decode_ids(preview_x[0].tolist()))
print(f"Input/target shape: {tuple(preview_x.shape)}; measured lessons: {preview_losses.numel()}")
print("PASS: targets are exactly one token ahead at every position")

exercise_length = min(32, model_config["context_length"])  # CHANGE THIS: try 16 or 48.
exercise_x, exercise_y = train_shard.sample(1, exercise_length, torch.Generator().manual_seed(SEED + 2))
shift_is_correct = torch.equal(exercise_x[:, 1:], exercise_y[:, :-1])
print(f"Your turn: length={exercise_length}; lessons={exercise_y.numel()}; shifted invariant={shift_is_correct}")
assert shift_is_correct

## 4 · The Practical Optimizer Loop

A raw update loop has three visible constraints. A full learning rate at step one can kick random weights too hard; an extreme gradient can dominate one update; and a useful effective batch may not fit in memory. Warmup-decay, clipping, and accumulation address those failures separately.

```mermaid
flowchart LR
    A["Microbatch 1 backward"] --> D["Accumulated gradients"]
    B["Microbatch 2 backward"] --> D
    D --> E{"Finite?"}
    E --> F["Clip global norm"]
    F --> G["AdamW step then LR schedule"]
    E --> H["Fail fast"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style H fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Warning:** clipping is not a repair for NaN or Inf. Non-finite logits, losses, gradients, or parameters stop the run immediately.

**Predict:** With two accumulation microbatches, how many optimizer steps should four backward passes produce? The counter asserts the answer during training.

In [ ]:
# ── Training Profile, Schedule, and State Utilities ─────────────────────────
@dataclass(frozen=True)
class TrainingProfile:
    optimizer_steps: int
    micro_batch_size: int
    accumulation_steps: int
    eval_every: int
    eval_batches: int
    learning_rate: float
    min_lr_ratio: float
    warmup_steps: int
    weight_decay: float
    clip_norm: float

profile = TrainingProfile(
    optimizer_steps=24 if QUICK_RUN else 200,
    micro_batch_size=2 if QUICK_RUN else 8,
    accumulation_steps=2 if QUICK_RUN else 4,
    eval_every=6 if QUICK_RUN else 20,
    eval_batches=4 if QUICK_RUN else 16,
    learning_rate=3e-4, min_lr_ratio=0.1,
    warmup_steps=3 if QUICK_RUN else 20,
    weight_decay=0.1, clip_norm=1.0,
)

def lr_multiplier(step: int) -> float:
    if step < profile.warmup_steps:
        return (step + 1) / max(1, profile.warmup_steps)
    progress = (step - profile.warmup_steps) / max(1, profile.optimizer_steps - profile.warmup_steps)
    cosine = 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))
    return profile.min_lr_ratio + (1.0 - profile.min_lr_ratio) * cosine

def make_optimizer(model_to_optimize: nn.Module):
    decay, no_decay = [], []
    for name, parameter in model_to_optimize.named_parameters():
        (no_decay if parameter.ndim < 2 or "norm" in name else decay).append(parameter)
    return torch.optim.AdamW(
        [{"params": decay, "weight_decay": profile.weight_decay},
         {"params": no_decay, "weight_decay": 0.0}],
        lr=profile.learning_rate, betas=(0.9, 0.95), eps=1e-8,
    )

def capture_rng(loader_generator: torch.Generator) -> dict:
    state = {
        "python": random.getstate(), "numpy": np.random.get_state(),
        "torch_cpu": torch.get_rng_state(), "loader": loader_generator.get_state(),
    }
    if torch.cuda.is_available():
        state["torch_cuda"] = torch.cuda.get_rng_state_all()
    return state

def restore_rng(state: dict, loader_generator: torch.Generator) -> None:
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch_cpu"])
    loader_generator.set_state(state["loader"])
    if torch.cuda.is_available() and "torch_cuda" in state:
        torch.cuda.set_rng_state_all(state["torch_cuda"])

def component_name(parameter_name: str) -> str:
    if parameter_name.startswith("token_embedding") or parameter_name.startswith("lm_head"):
        return "tied embedding/head"
    if parameter_name.startswith("blocks."):
        return ".".join(parameter_name.split(".")[:2])
    return "final norm"

def assert_finite_model(model_to_check: nn.Module) -> None:
    for name, parameter in model_to_check.named_parameters():
        assert_finite_tensor(f"parameter {name}", parameter)
        if parameter.grad is not None:
            assert_finite_tensor(f"gradient {name}", parameter.grad)

def global_grad_norm(model_to_check: nn.Module) -> float:
    squared = sum(
        parameter.grad.detach().float().pow(2).sum()
        for parameter in model_to_check.parameters() if parameter.grad is not None
    )
    return float(squared.sqrt())

def gradient_norms_by_component(model_to_check: nn.Module) -> dict[str, float]:
    totals: dict[str, float] = {}
    for name, parameter in model_to_check.named_parameters():
        if parameter.grad is not None:
            key = component_name(name)
            totals[key] = totals.get(key, 0.0) + float(parameter.grad.detach().float().pow(2).sum())
    return {key: math.sqrt(value) for key, value in totals.items()}

**Code Walkthrough**

The quick profile performs 24 optimizer updates from 48 microbatches. The larger profile changes budget, not objective. RNG capture includes Python, NumPy, CPU Torch, the data-loader generator, and CUDA generators when present. Parameters with matrix-shaped weights receive AdamW decay; norms and scalar-like parameters do not.

In [ ]:
# ── Evaluation and Complete Checkpoint I/O ─────────────────────────────────
@torch.no_grad()
def evaluate(model_to_evaluate: TinyModernLM, shard: PackedShard):
    model_to_evaluate.eval()
    weighted_loss, token_count = 0.0, 0
    position_totals = torch.zeros(sequence_length, device=DEVICE)
    fixed_x, fixed_y = shard.fixed(profile.micro_batch_size * profile.eval_batches, sequence_length)
    for batch_index in range(profile.eval_batches):
        start = batch_index * profile.micro_batch_size
        x = fixed_x[start:start + profile.micro_batch_size]
        y = fixed_y[start:start + profile.micro_batch_size]
        logits, _ = model_to_evaluate(x)
        assert_finite_tensor("validation logits", logits)
        losses = F.cross_entropy(logits.transpose(1, 2), y, reduction="none")
        assert_finite_tensor("validation loss", losses)
        weighted_loss += float(losses.sum())
        token_count += losses.numel()
        position_totals += losses.sum(dim=0)
    model_to_evaluate.train()
    return weighted_loss / token_count, (position_totals / token_count * sequence_length).cpu().numpy()

def checkpoint_manifest(step: int) -> dict:
    return {
        "schema_version": 1, "optimizer_step": step, "seed": SEED,
        "config_digest": identities["config_digest"],
        "dataset_manifest_digest": identities["dataset_manifest_digest"],
        "train_sha256": identities["train_sha256"],
        "validation_sha256": identities["validation_sha256"],
    }

def save_checkpoint(directory: Path, model_to_save: TinyModernLM, optimizer, scheduler,
                    loader_generator: torch.Generator, step: int, microbatches_seen: int) -> None:
    directory.mkdir(parents=True, exist_ok=True)
    torch.save(model_to_save.state_dict(), directory / "model.pt")
    torch.save(optimizer.state_dict(), directory / "optimizer.pt")
    torch.save(scheduler.state_dict(), directory / "scheduler.pt")
    torch.save(capture_rng(loader_generator), directory / "rng.pt")
    (directory / "counters.json").write_text(
        json.dumps({"optimizer_step": step, "microbatches_seen": microbatches_seen}, indent=2) + "\n",
        encoding="utf-8",
    )
    (directory / "checkpoint-manifest.json").write_text(
        json.dumps(checkpoint_manifest(step), indent=2) + "\n", encoding="utf-8"
    )

def load_checkpoint(directory: Path, model_to_load: TinyModernLM, optimizer, scheduler,
                    loader_generator: torch.Generator) -> dict:
    saved_manifest = json.loads((directory / "checkpoint-manifest.json").read_text(encoding="utf-8"))
    current = checkpoint_manifest(saved_manifest["optimizer_step"])
    if any(saved_manifest.get(key) != current[key] for key in current):
        raise ValueError("Checkpoint refusal: config or dataset identity changed since this state was saved")
    model_to_load.load_state_dict(torch.load(directory / "model.pt", map_location=DEVICE, weights_only=True))
    optimizer.load_state_dict(torch.load(directory / "optimizer.pt", map_location=DEVICE, weights_only=True))
    scheduler.load_state_dict(torch.load(directory / "scheduler.pt", map_location=DEVICE, weights_only=True))
    restore_rng(torch.load(directory / "rng.pt", map_location="cpu", weights_only=False), loader_generator)
    return json.loads((directory / "counters.json").read_text(encoding="utf-8"))

In [ ]:
# ── Sparse-Snapshot Training Loop ──────────────────────────────────────────
def train_base_model(model_to_train: TinyModernLM):
    optimizer = make_optimizer(model_to_train)
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_multiplier)
    loader_generator = torch.Generator().manual_seed(SEED + 10)
    history = {key: [] for key in ["step", "train_loss", "validation_loss", "lr", "grad_before", "grad_after"]}
    snapshots, checkpoint_records, parity_capture = [], [], None
    microbatches_seen = optimizer_steps_seen = 0
    initial_validation, initial_positions = evaluate(model_to_train, validation_shard)
    snapshots.append({"step": 0, "position_loss": initial_positions, "gradients": {}, "continuation": initial_generation})
    start_time = time.perf_counter()
    if DEVICE.type == "cpu":
        tracemalloc.start()
    else:
        torch.cuda.reset_peak_memory_stats()

    for optimizer_step in range(1, profile.optimizer_steps + 1):
        optimizer.zero_grad(set_to_none=True)
        accumulated_loss = 0.0
        for _ in range(profile.accumulation_steps):
            x, y = train_shard.sample(profile.micro_batch_size, sequence_length, loader_generator)
            logits, loss = model_to_train(x, y)
            assert loss is not None
            assert_finite_tensor("training logits", logits)
            assert_finite_tensor("training loss", loss)
            (loss / profile.accumulation_steps).backward()
            accumulated_loss += float(loss.detach())
            microbatches_seen += 1
        assert_finite_model(model_to_train)
        before = global_grad_norm(model_to_train)
        gradients = gradient_norms_by_component(model_to_train)
        torch.nn.utils.clip_grad_norm_(model_to_train.parameters(), profile.clip_norm, error_if_nonfinite=True)
        after = global_grad_norm(model_to_train)
        optimizer.step()
        scheduler.step()
        optimizer_steps_seen += 1
        assert optimizer_steps_seen * profile.accumulation_steps == microbatches_seen
        assert_finite_model(model_to_train)
        history["step"].append(optimizer_step)
        history["train_loss"].append(accumulated_loss / profile.accumulation_steps)
        history["lr"].append(optimizer.param_groups[0]["lr"])
        history["grad_before"].append(before)
        history["grad_after"].append(after)

        if optimizer_step % profile.eval_every == 0 or optimizer_step == profile.optimizer_steps:
            validation_loss, position_loss = evaluate(model_to_train, validation_shard)
            history["validation_loss"].append({"step": optimizer_step, "loss": validation_loss})
            continuation = decode_ids(model_to_train.generate(probe_ids, max_new_tokens=12))
            checkpoint_dir = CHECKPOINT_ROOT / f"step-{optimizer_step:06d}"
            save_checkpoint(checkpoint_dir, model_to_train, optimizer, scheduler,
                            loader_generator, optimizer_step, microbatches_seen)
            checkpoint_records.append({"step": optimizer_step, "validation_loss": validation_loss,
                                       "path": str(checkpoint_dir)})
            snapshots.append({"step": optimizer_step, "position_loss": position_loss,
                              "gradients": gradients, "continuation": continuation})
            if parity_capture is None:
                temporary_generator = torch.Generator()
                temporary_generator.set_state(loader_generator.get_state())
                probe_microbatches = [
                    train_shard.sample(profile.micro_batch_size, sequence_length, temporary_generator)
                    for _ in range(profile.accumulation_steps)
                ]
                parity_capture = {
                    "checkpoint": checkpoint_dir,
                    "model": copy.deepcopy(model_to_train.state_dict()),
                    "optimizer": copy.deepcopy(optimizer.state_dict()),
                    "scheduler": copy.deepcopy(scheduler.state_dict()),
                    "rng": copy.deepcopy(capture_rng(loader_generator)),
                    "microbatches": probe_microbatches,
                }
            print(f"step={optimizer_step:3d} train={history['train_loss'][-1]:.4f} "
                  f"validation={validation_loss:.4f} grad={before:.3f}->{after:.3f}")

    runtime = time.perf_counter() - start_time
    if DEVICE.type == "cuda":
        memory = {"type": "cuda_measured", "bytes": int(torch.cuda.max_memory_allocated())}
    else:
        _, peak = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        memory = {"type": "cpu_tracemalloc", "bytes": int(peak)}
    run_state = {
        "history": history, "snapshots": snapshots, "checkpoints": checkpoint_records,
        "parity_capture": parity_capture, "runtime_seconds": runtime, "peak_memory": memory,
        "initial_validation_loss": initial_validation, "microbatches_seen": microbatches_seen,
        "optimizer_steps_seen": optimizer_steps_seen,
    }
    return run_state

run_state = train_base_model(model)
print(f"PASS: {run_state['microbatches_seen']} microbatches produced exactly {run_state['optimizer_steps_seen']} optimizer steps")
print(f"Measured runtime: {run_state['runtime_seconds']:.1f}s; peak memory label: {run_state['peak_memory']['type']}")

**Code Walkthrough: the complete training state machine**

1. `optimizer.zero_grad(set_to_none=True)` starts one optimizer update with empty gradient buffers. The next `accumulation_steps` microbatches divide their losses before backpropagation, so their gradients sum to one effective batch rather than multiplying the update size.
2. The microbatch counter and optimizer-step counter are tracked separately. Their final assertion proves that an optimizer update occurs only after the configured number of backward passes.
3. Before every step, the loop rejects non-finite loss and gradients. It records the global norm before clipping and the bounded norm after clipping, then lets AdamW update parameters and the scheduler advance once.
4. Lightweight history is recorded every optimizer step: train loss, learning rate, and clipping measurements. Expensive diagnostics are sparse and occur only at `eval_every` checkpoints.
5. Each checkpoint evaluates the held-out shard token by token, records a per-position loss profile, captures gradient norms by block, and generates the same fixed probe. The later animations therefore use real snapshots rather than interpolated frames.
6. The first interval checkpoint also captures model, optimizer, scheduler, loader RNG state, and deterministic probe microbatches. Section 6 reloads that complete state and compares its next update with an uninterrupted reference update.
7. CPU runs use `tracemalloc`; CUDA runs use PyTorch's allocator measurement. The manifest labels which measurement produced the number instead of presenting an estimate as observed memory.

**Checkpoint:** Two microbatches feed exactly one optimizer update, validation remains separate from optimization, and every plotted training value comes from the actual run.

## 5 · Watch Predictions and Training State Change

A final loss number hides too much. Sparse snapshots let you inspect where validation loss moved, which blocks received gradient, and whether a fixed continuation changed without pretending that a tiny run acquired broad knowledge.

```mermaid
flowchart LR
    A["Step 0 snapshot"] --> B["Checkpoint 1"] --> C["Checkpoint 2"] --> D["Final checkpoint"]
    A --> E["Position loss"]
    B --> F["Block gradients"]
    C --> G["Fixed prompt"]
    D --> H["Train and validation curves"]
    style A fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style H fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

Before each animation, read the printed instruction. Frames are checkpoint measurements, not interpolated values.

In [ ]:
# ── Measured Distributions, Curves, Schedule, Clipping, and Accumulation ────
validation_records = run_state["history"]["validation_loss"]
selected_record = min(run_state["checkpoints"], key=lambda record: record["validation_loss"])
selected_step = selected_record["step"]
selected_model = TinyModernLM(model_config).to(DEVICE)
selected_model.load_state_dict(torch.load(Path(selected_record["path"]) / "model.pt", map_location=DEVICE, weights_only=True))
selected_model.eval()
with torch.no_grad():
    trained_logits, _ = selected_model(probe_tensor)
trained_distribution = trained_logits[0, -1].softmax(-1).cpu()

def top_distribution(distribution: torch.Tensor, count: int = 10):
    values, ids = distribution.topk(count)
    labels = [decode_ids([int(token_id)]) or f"id:{int(token_id)}" for token_id in ids]
    return labels, values.numpy()

initial_labels, initial_values = top_distribution(initial_distribution)
trained_labels, trained_values = top_distribution(trained_distribution)
fig, axes = plt.subplots(2, 2, figsize=(14, 9), facecolor=PLOT_BG)
for axis in axes.flat:
    axis.set_facecolor(PLOT_BG)
    axis.tick_params(colors="white")
    axis.title.set_color("white")
axes[0, 0].barh(initial_labels[::-1], initial_values[::-1], color=COLORS[0])
axes[0, 0].set_title("Initial next-token distribution: top 10")
axes[0, 1].barh(trained_labels[::-1], trained_values[::-1], color=COLORS[2])
axes[0, 1].set_title(f"Selected-checkpoint distribution: step {selected_step}")
axes[1, 0].plot(run_state["history"]["step"], run_state["history"]["train_loss"], color=COLORS[0], label="train")
axes[1, 0].plot([item["step"] for item in validation_records], [item["loss"] for item in validation_records], marker="o", color=COLORS[1], label="validation")
axes[1, 0].axvline(selected_step, color=COLORS[2], linestyle="--", label="selected by validation")
axes[1, 0].set_title("Measured train and held-out validation loss")
axes[1, 0].legend(facecolor=PLOT_BG, labelcolor="white")
axes[1, 1].plot(run_state["history"]["step"], run_state["history"]["lr"], color=COLORS[4])
axes[1, 1].axvspan(1, profile.warmup_steps, color=COLORS[1], alpha=0.25, label="warmup")
axes[1, 1].axvspan(profile.warmup_steps, profile.optimizer_steps, color=COLORS[0], alpha=0.12, label="decay")
axes[1, 1].set_title("Applied learning-rate schedule")
axes[1, 1].legend(facecolor=PLOT_BG, labelcolor="white")
fig.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 4), facecolor=PLOT_BG)
steps = run_state["history"]["step"]
axes[0].plot(steps, run_state["history"]["grad_before"], label="before", color=COLORS[3])
axes[0].plot(steps, run_state["history"]["grad_after"], label="after", color=COLORS[2])
axes[0].axhline(profile.clip_norm, color="white", linestyle=":", label="clip threshold")
axes[0].set_title("Measured clipping before and after", color="white")
axes[0].legend(facecolor=PLOT_BG, labelcolor="white")
for microbatch in range(profile.accumulation_steps * 3):
    update = microbatch // profile.accumulation_steps
    axes[1].barh(update, 0.8, left=microbatch, color=COLORS[update])
    axes[1].text(microbatch + 0.4, update, f"micro {microbatch + 1}", ha="center", va="center", fontsize=8)
axes[1].set_title("Gradient-accumulation timeline", color="white")
axes[1].set_xlabel("Backward-pass order", color="white")
for axis in axes:
    axis.set_facecolor(PLOT_BG)
    axis.tick_params(colors="white")
fig.tight_layout()
plt.show()
print("PASS: plots use real distributions, losses, learning rates, and gradient norms from this run")

In [ ]:
# ── Sparse Checkpoint Animations ───────────────────────────────────────────
def show_position_loss_animation(snapshots: list[dict]) -> None:
    fig, axis = plt.subplots(figsize=(10, 4), facecolor=PLOT_BG)
    axis.set_facecolor(PLOT_BG)
    maximum = max(float(np.max(snapshot["position_loss"])) for snapshot in snapshots)
    bars = axis.bar(np.arange(sequence_length), snapshots[0]["position_loss"], color=COLORS[1])
    axis.set_ylim(0, maximum * 1.1)
    axis.set_xlabel("Token position", color="white")
    axis.set_ylabel("Held-out loss", color="white")
    axis.tick_params(colors="white")
    def update(frame: int):
        for bar, height in zip(bars, snapshots[frame]["position_loss"]):
            bar.set_height(float(height))
        axis.set_title(f"Per-position validation loss at step {snapshots[frame]['step']}", color="white")
        return tuple(bars)
    animation = FuncAnimation(fig, update, frames=len(snapshots), interval=900, blit=False)
    plt.close(fig)
    print("Watch for positions that remain difficult; frames are measured checkpoints only.")
    display(HTML(animation.to_jshtml()))

def show_gradient_animation(snapshots: list[dict]) -> None:
    measured = [snapshot for snapshot in snapshots if snapshot["gradients"]]
    components = sorted({name for snapshot in measured for name in snapshot["gradients"]})
    fig, axis = plt.subplots(figsize=(10, 4), facecolor=PLOT_BG)
    axis.set_facecolor(PLOT_BG)
    maximum = max(snapshot["gradients"].get(name, 0.0) for snapshot in measured for name in components)
    bars = axis.bar(components, [measured[0]["gradients"].get(name, 0.0) for name in components], color=COLORS[0])
    axis.set_ylim(0, maximum * 1.15 if maximum else 1.0)
    axis.tick_params(axis="x", rotation=25, colors="white")
    axis.tick_params(axis="y", colors="white")
    def update(frame: int):
        for bar, name in zip(bars, components):
            bar.set_height(measured[frame]["gradients"].get(name, 0.0))
        axis.set_title(f"Gradient norm by component at step {measured[frame]['step']}", color="white")
        return tuple(bars)
    animation = FuncAnimation(fig, update, frames=len(measured), interval=900, blit=False)
    plt.close(fig)
    print("Watch every Transformer block receive finite gradient; magnitude need not be equal.")
    display(HTML(animation.to_jshtml()))

def show_continuation_animation(snapshots: list[dict]) -> None:
    fig, axis = plt.subplots(figsize=(12, 2.8), facecolor=PLOT_BG)
    axis.set_facecolor(PLOT_BG)
    axis.axis("off")
    text = axis.text(0.02, 0.55, "", color="white", fontsize=12, wrap=True, va="center")
    def update(frame: int):
        snapshot = snapshots[frame]
        text.set_text(snapshot["continuation"])
        axis.set_title(f"Fixed greedy continuation at step {snapshot['step']}", color="white")
        return (text,)
    animation = FuncAnimation(fig, update, frames=len(snapshots), interval=1200, blit=False)
    plt.close(fig)
    print("Watch the same prompt change; odd text is an honest tiny-run result, not a capability claim.")
    display(HTML(animation.to_jshtml()))

show_position_loss_animation(run_state["snapshots"])
show_gradient_animation(run_state["snapshots"])
show_continuation_animation(run_state["snapshots"])

**Code Walkthrough**

The top-token panels compare the same fixed prompt before and after the validation-selected checkpoint. The curves do not assume validation improves monotonically. Every animation uses `FuncAnimation`, closes its figure before display, renders with `to_jshtml`, and uses measured checkpoint frames only.

## 6 · Checkpoint and Resume: Weights Alone Are Not Training State

A model-only save can generate, but it cannot faithfully continue AdamW training. The next update also depends on moments, LR position, counters, data sampling, and RNG state.

```mermaid
flowchart LR
    A["Live step k"] --> B["Complete checkpoint"]
    A --> C["In-memory reference next update"]
    B --> D["Fresh objects plus restore"]
    D --> E["Resumed next update"]
    C --> F{"Loss and parameters within tolerance?"}
    E --> F
    F --> G["Resume contract passes"]
    F --> H["Refuse reproducibility claim"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style H fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

The check uses the same hardware and a realistic tolerance: `1e-6` on CPU and `1e-5` on CUDA. It does not promise bitwise parity across different devices or software versions.

In [ ]:
# ── One-Update Resume Parity and Timeline ──────────────────────────────────
def one_update(model_for_step: TinyModernLM, optimizer, scheduler, microbatches) -> float:
    model_for_step.train()
    optimizer.zero_grad(set_to_none=True)
    losses = []
    for x, y in microbatches:
        logits, loss = model_for_step(x, y)
        assert loss is not None
        assert_finite_tensor("resume probe logits", logits)
        assert_finite_tensor("resume probe loss", loss)
        (loss / len(microbatches)).backward()
        losses.append(float(loss.detach()))
    torch.nn.utils.clip_grad_norm_(model_for_step.parameters(), profile.clip_norm, error_if_nonfinite=True)
    optimizer.step()
    scheduler.step()
    return sum(losses) / len(losses)

def build_training_bundle():
    candidate = TinyModernLM(model_config).to(DEVICE)
    optimizer = make_optimizer(candidate)
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_multiplier)
    generator = torch.Generator().manual_seed(0)
    return candidate, optimizer, scheduler, generator

capture = run_state["parity_capture"]
reference_model, reference_optimizer, reference_scheduler, reference_generator = build_training_bundle()
reference_model.load_state_dict(capture["model"])
reference_optimizer.load_state_dict(capture["optimizer"])
reference_scheduler.load_state_dict(capture["scheduler"])
restore_rng(capture["rng"], reference_generator)
reference_loss = one_update(reference_model, reference_optimizer, reference_scheduler, capture["microbatches"])

resumed_model, resumed_optimizer, resumed_scheduler, resumed_generator = build_training_bundle()
resume_counters = load_checkpoint(capture["checkpoint"], resumed_model, resumed_optimizer, resumed_scheduler, resumed_generator)
resumed_loss = one_update(resumed_model, resumed_optimizer, resumed_scheduler, capture["microbatches"])
max_parameter_delta = max(
    float((left - right).abs().max())
    for left, right in zip(reference_model.parameters(), resumed_model.parameters())
)
tolerance = 1e-6 if DEVICE.type == "cpu" else 1e-5
loss_delta = abs(reference_loss - resumed_loss)
assert loss_delta <= tolerance and max_parameter_delta <= tolerance
resume_parity = {
    "checkpoint_step": resume_counters["optimizer_step"],
    "reference_loss": reference_loss, "resumed_loss": resumed_loss,
    "loss_delta": loss_delta, "max_parameter_delta": max_parameter_delta,
    "tolerance": tolerance,
}
print(json.dumps(resume_parity, indent=2))
print("PASS: complete checkpoint restoration matches the same-hardware in-memory next update")

fig, axis = plt.subplots(figsize=(11, 3), facecolor=PLOT_BG)
axis.set_facecolor(PLOT_BG)
checkpoint_step = resume_parity["checkpoint_step"]
axis.plot([0, checkpoint_step, checkpoint_step + 1], [1.05] * 3, color=COLORS[0], linewidth=5, label="in-memory reference")
axis.plot([checkpoint_step, checkpoint_step + 1], [0.75] * 2, color=COLORS[2], linewidth=5, label="restored checkpoint")
axis.scatter([checkpoint_step], [0.9], color=COLORS[1], s=120, label="saved complete state")
axis.text(checkpoint_step + 0.5, 0.62, f"next-loss delta={loss_delta:.2e}", color="white", ha="center")
axis.set_ylim(0.45, 1.3)
axis.set_xlabel("Optimizer step", color="white")
axis.set_yticks([])
axis.tick_params(colors="white")
axis.set_title("Measured same-hardware resume parity", color="white")
axis.legend(facecolor=PLOT_BG, labelcolor="white")
plt.show()

## 7 · Select by Held-Out Validation, Then Interpret the Actual Result

The latest checkpoint is not automatically the best checkpoint. Riverside chooses among interval checkpoints using one field only: token-weighted validation loss. Training loss and prompt appearance are excluded from selection.

```mermaid
flowchart LR
    A["Interval checkpoints"] --> B["Held-out validation loss only"]
    B --> C["Minimum measured loss"]
    C --> D["Selected base checkpoint"]
    E["Train loss"] --> F["Diagnostic only"]
    G["Prompt text"] --> F
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Warning:** This construction lab has no test-set capability evaluation. Validation selects training state; it does not prove broad knowledge, semantics, reasoning, or assistant behavior.

In [ ]:
# ── Validation-Only Selection and Honest Branch ─────────────────────────────
selection_inputs = [
    {"step": record["step"], "validation_loss": record["validation_loss"]}
    for record in run_state["checkpoints"]
]
selected_by_validation = min(selection_inputs, key=lambda record: record["validation_loss"])
assert set(selected_by_validation) == {"step", "validation_loss"}
assert selected_by_validation["step"] == selected_step
initial_validation = run_state["initial_validation_loss"]
final_validation = selected_by_validation["validation_loss"]
improvement = initial_validation - final_validation
MEANINGFUL_QUICK_RUN_DELTA = 0.01
print(f"Initial held-out loss: {initial_validation:.4f}")
print(f"Selected held-out loss: {final_validation:.4f} at step {selected_step}")
if improvement >= MEANINGFUL_QUICK_RUN_DELTA:
    print(f"Measured result: validation loss improved by {improvement:.4f} on this frozen split.")
    print("Interpretation: the tiny run learned some local next-token regularity; no broader capability follows.")
elif improvement > 0:
    print(f"Measured result: validation loss moved down by only {improvement:.4f}.")
    print("Interpretation: the direction is encouraging, but this quick-run delta is too small to separate from run noise.")
else:
    print(f"Measured result: validation did not improve; delta={improvement:.4f}.")
    print("Interpretation: this quick run is too small or unstable for held-out improvement. The checkpoint remains a pipeline artifact, not a success claim.")
print("PASS: checkpoint selection consumed validation loss only")

## 8 · Crack Open What Changed

A falling scalar cannot tell you whether every trainable surface moved. Compare the validation-selected state with the original random state by component, then inspect nearest embedding vectors for frequent probe tokens. Neighbor movement is only a training diagnostic.

```mermaid
flowchart LR
    A["Initial random state"] --> C["Parameter difference by component"]
    B["Selected checkpoint"] --> C
    C --> D{"Every expected surface changed?"}
    B --> E["Embedding neighbors"]
    E --> F["Diagnostic, not semantics"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Predict:** Tied embedding and LM-head names point to one stored parameter. Should they be counted twice in the update chart? The grouping below counts that storage once.

In [ ]:
# ── Parameter Updates and Embedding Neighbors ───────────────────────────────
selected_state = selected_model.state_dict()
component_squared: dict[str, float] = {}
changed_components: set[str] = set()
for name, final_value in selected_state.items():
    if name == "lm_head.weight":
        continue  # tied storage is represented by token_embedding.weight
    key = component_name(name)
    difference = final_value.detach().cpu().float() - initial_state[name].float()
    component_squared[key] = component_squared.get(key, 0.0) + float(difference.pow(2).sum())
    if float(difference.abs().max()) > 0:
        changed_components.add(key)
component_updates = {key: math.sqrt(value) for key, value in component_squared.items()}
expected_components = {
    "tied embedding/head", "final norm",
    *{f"blocks.{index}" for index in range(model_config["n_layers"])},
}
assert expected_components <= changed_components
assert selected_model.token_embedding.weight.data_ptr() == selected_model.lm_head.weight.data_ptr()

fig, axis = plt.subplots(figsize=(10, 4), facecolor=PLOT_BG)
axis.set_facecolor(PLOT_BG)
axis.bar(list(component_updates), list(component_updates.values()), color=COLORS[0])
axis.set_ylabel("L2 update magnitude", color="white")
axis.set_title("Measured parameter update by component", color="white")
axis.tick_params(axis="x", rotation=25, colors="white")
axis.tick_params(axis="y", colors="white")
plt.show()

def nearest_tokens(weight: torch.Tensor, token_id: int, count: int = 5):
    normalized = F.normalize(weight.float(), dim=-1)
    similarities = normalized @ normalized[token_id]
    similarities[token_id] = -1
    values, ids = similarities.topk(count)
    return [
        (decode_ids([int(index)]) or f"id:{int(index)}", float(value))
        for value, index in zip(values, ids)
    ]

initial_embedding = initial_state["token_embedding.weight"]
final_embedding = selected_state["token_embedding.weight"].detach().cpu()
for word in ["Aria", "signal", "Meridian"]:
    encoded = tokenizer.encode(word).ids
    if not encoded:
        continue
    token_id = encoded[0]
    print(f"{word!r} first-piece id={token_id}")
    print("  initial:", nearest_tokens(initial_embedding.clone(), token_id))
    print("  selected:", nearest_tokens(final_embedding.clone(), token_id))
print("PASS: every expected component changed and tied storage remains tied")
print("Warning: neighbor movement measures geometry, not semantic understanding")

## 9 · Same Mechanism, Larger Budget

The objective does not change at 10M, 100M, 1B, or 7B parameters. Capacity, tokens per update, memory, and compute do. The calculator labels all scale values as estimates; only this notebook's runtime memory was measured.

```mermaid
flowchart LR
    A["Tiny next-token loop"] --> B["10M"] --> C["100M"] --> D["1B"] --> E["7B"]
    A --> F["Same tokenizer-to-loss path"]
    E --> G["Distributed training infrastructure"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

The AI Infrastructure track takes over when one process can no longer hold weights, gradients, optimizer state, and activations. This notebook stops at the accounting boundary.

In [ ]:
# ── Parameter, Memory, Token, and Compute Scale Calculator ─────────────────
scale_profiles = [
    {"label": "10M", "parameters": 10_000_000, "tokens_per_step": 32_768},
    {"label": "100M", "parameters": 100_000_000, "tokens_per_step": 131_072},
    {"label": "1B", "parameters": 1_000_000_000, "tokens_per_step": 524_288},
    {"label": "7B", "parameters": 7_000_000_000, "tokens_per_step": 2_097_152},
]
for item in scale_profiles:
    parameters = item["parameters"]
    item["weights_gib"] = parameters * 2 / 2**30
    item["gradients_gib"] = parameters * 2 / 2**30
    item["optimizer_gib"] = parameters * 8 / 2**30
    item["rough_flops_per_step"] = 6 * parameters * item["tokens_per_step"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor=PLOT_BG)
labels = [item["label"] for item in scale_profiles]
bottom = np.zeros(len(labels))
for field, label, color in [
    ("weights_gib", "weights", COLORS[0]),
    ("gradients_gib", "gradients", COLORS[1]),
    ("optimizer_gib", "Adam moments", COLORS[3]),
]:
    values = np.array([item[field] for item in scale_profiles])
    axes[0].bar(labels, values, bottom=bottom, label=f"{label} (estimated)", color=color)
    bottom += values
axes[0].set_yscale("log")
axes[0].set_ylabel("GiB, log scale", color="white")
axes[0].set_title("Estimated persistent training-state memory", color="white")
axes[0].legend(facecolor=PLOT_BG, labelcolor="white")
axes[1].bar(labels, [item["parameters"] for item in scale_profiles], color=COLORS[2])
axes[1].set_yscale("log")
axes[1].set_ylabel("Parameters, log scale", color="white")
axes[1].set_title("Target parameter bars", color="white")
for axis in axes:
    axis.set_facecolor(PLOT_BG)
    axis.tick_params(colors="white")
fig.tight_layout()
plt.show()
for item in scale_profiles:
    print(f"{item['label']:>4}: tokens/step={item['tokens_per_step']:,}; rough FLOPs/step={item['rough_flops_per_step']:.3e}; values are estimates")
print(f"This run's peak-memory value is measured separately: {run_state['peak_memory']}")

## 10 · Package and Reload the Final Base Checkpoint

The selected checkpoint becomes useful only when a clean object can reconstruct it with the same config and tokenizer identity. Packaging copies the selected model state, its optimizer/scheduler continuation state, config, validated tokenizer directory, and a complete training manifest.

```mermaid
flowchart TD
    A["artifacts/base-lm"] --> B["checkpoints/step-*"]
    A --> C["final"]
    C --> D["model.pt"]
    C --> E["optimizer.pt + scheduler.pt"]
    C --> F["model-config.json"]
    C --> G["tokenizer"]
    C --> H["training-manifest.json"]
    C --> I["rng.pt + counters.json"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style H fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style I fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Checkpoint:** Runtime writes remain under `artifacts/base-lm/checkpoints` and `artifacts/base-lm/final`, except the contract-required root `training-manifest.json` and the explicitly permitted fallback `model-config.json`.

In [ ]:
# ── Final Artifact Manifest and Package ─────────────────────────────────────
selected_checkpoint = Path(selected_record["path"])
if FINAL_ROOT.exists():
    shutil.rmtree(FINAL_ROOT)
FINAL_ROOT.mkdir(parents=True, exist_ok=True)
for filename in [
    "model.pt", "optimizer.pt", "scheduler.pt", "rng.pt",
    "counters.json", "checkpoint-manifest.json",
]:
    shutil.copy2(selected_checkpoint / filename, FINAL_ROOT / filename)
shutil.copy2(ARTIFACT_ROOT / "model-config.json", FINAL_ROOT / "model-config.json")
shutil.copytree(ARTIFACT_ROOT / "tokenizer", FINAL_ROOT / "tokenizer")
final_tokenizer_digest = sha256_file(FINAL_ROOT / "tokenizer" / "tokenizer.json")
assert final_tokenizer_digest == identities["tokenizer_digest"]

training_manifest = {
    "schema_version": 1,
    "seed": SEED,
    "package_versions": {
        "python": platform.python_version(), "torch": torch.__version__, "numpy": np.__version__,
    },
    "hardware_profile": {
        "device": str(DEVICE), "platform": platform.platform(),
        "cuda_available": torch.cuda.is_available(),
    },
    "config_digest": identities["config_digest"],
    "dataset_manifest_digest": identities["dataset_manifest_digest"],
    "tokenizer_digest": identities["tokenizer_digest"],
    "train_sha256": identities["train_sha256"],
    "validation_sha256": identities["validation_sha256"],
    "train_tokens": int(dataset_manifest["train_tokens"]),
    "validation_tokens": int(dataset_manifest["validation_tokens"]),
    "optimizer_settings": asdict(profile),
    "checkpoint_steps": [record["step"] for record in run_state["checkpoints"]],
    "selected_checkpoint": str(selected_checkpoint),
    "selection_metric": "token_weighted_validation_loss",
    "final_validation_loss": float(final_validation),
    "final_validation_perplexity": float(math.exp(min(20.0, final_validation))),
    "runtime_seconds": float(run_state["runtime_seconds"]),
    "peak_memory_measurement_type": run_state["peak_memory"]["type"],
    "peak_memory_bytes": run_state["peak_memory"]["bytes"],
    "quick_run": QUICK_RUN,
    "pretrained_weights_loaded": False,
    "resume_parity": resume_parity,
    "scope": "base_model_only_no_sft_rlhf_rag_assistant_or_serving",
}
manifest_text = json.dumps(training_manifest, indent=2) + "\n"
(ARTIFACT_ROOT / "training-manifest.json").write_text(manifest_text, encoding="utf-8")
(FINAL_ROOT / "training-manifest.json").write_text(manifest_text, encoding="utf-8")
print(f"Packaged selected checkpoint step {selected_step} in {FINAL_ROOT}")
print("PASS: final tokenizer digest matches the validated source tokenizer")

In [ ]:
# ── Clean Reload, Generation Parity, and Artifact Tree ──────────────────────
reloaded_config = json.loads((FINAL_ROOT / "model-config.json").read_text(encoding="utf-8"))
reloaded_config["vocab_size"] = dataset_manifest["vocab_size"]
reloaded_tokenizer = Tokenizer.from_file(str(FINAL_ROOT / "tokenizer" / "tokenizer.json"))
reloaded_model = TinyModernLM(reloaded_config).to(DEVICE)
reloaded_model.load_state_dict(torch.load(FINAL_ROOT / "model.pt", map_location=DEVICE, weights_only=True))
reloaded_model.eval()
with torch.no_grad():
    selected_logits, _ = selected_model(probe_tensor)
    reloaded_logits, _ = reloaded_model(probe_tensor)
reload_max_delta = float((selected_logits - reloaded_logits).abs().max())
selected_generation_ids = selected_model.generate(probe_ids, max_new_tokens=12)
reloaded_generation_ids = reloaded_model.generate(probe_ids, max_new_tokens=12)
assert reload_max_delta <= tolerance
assert selected_generation_ids == reloaded_generation_ids
assert reloaded_model.token_embedding.weight.data_ptr() == reloaded_model.lm_head.weight.data_ptr()
print(f"Reload max-logit delta: {reload_max_delta:.3e}")
print(f"Reloaded continuation: {reloaded_tokenizer.decode(reloaded_generation_ids, skip_special_tokens=False)!r}")
print("PASS: clean reload reproduces fixed logits, greedy generation, and tied storage")

print("\nArtifact tree:")
for path in sorted(FINAL_ROOT.rglob("*")):
    if path.is_file():
        print(f"  {path.relative_to(ARTIFACT_ROOT)} ({path.stat().st_size:,} bytes)")

## 11 · Scorecard, Reflection, and Scope Ledger

```mermaid
flowchart LR
    A["Notebook 07 manifest plus shard digests"] --> B["Validated tokenizer and packed data"]
    C["Notebook 06 config or announced fallback"] --> D["Random TinyModernLM"]
    B --> E["AdamW next-token training"]
    D --> E
    E --> F["Interval validation checkpoints"]
    F --> G["Minimum validation loss"]
    G --> H["Final base-model package"]
    H --> I["Clean reload parity"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style H fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style I fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

### Completed Roadmap

| Constraint | Before | After |
| --- | --- | --- |
| Artifact identity | files could be assumed | schemas and SHA-256 values are verified before model construction |
| Predictions | seeded random continuation | measured selected-checkpoint distribution and continuation |
| Update control | no optimizer state | AdamW, warmup-decay, clipping, and accumulation are measured |
| Validation | no held-out selection | interval checkpoints are ranked by token-weighted validation loss only |
| Reproducibility | weights alone | optimizer, scheduler, RNG, counters, and digests resume within same-hardware tolerance |
| Packaging | no reusable artifact | final config, tokenizer, states, manifest, and clean reload parity |

### Scorecard

- Data/config identity checks: measured before training.
- Finite logits, losses, gradients, and parameters: fail-fast checks active.
- Expected gradient and parameter-update surfaces: verified.
- Sparse animations: actual checkpoint frames only.
- Result interpretation: branched from measured validation change.
- Memory: runtime measurement labeled separately from scale estimates.

### Reflection

You did not change the architecture into something magical. You kept asking the same local next-token question, applied controlled updates, checked held-out loss, and preserved enough state to reproduce the boundary. The useful achievement is the audited pipeline and artifact, not a claim that a tiny model understands Riverside.

### Three-Tier Ledger

| Tier | Coverage |
| --- | --- |
| Implemented and demonstrated | strict manifests/digests; random TinyModernLM; packed shifted targets; AdamW; warmup-decay; clipping; accumulation; sparse snapshots; held-out validation; complete checkpoint/RNG resume; validation selection; parameter diffs; embedding-neighbor diagnostic; scale calculator; final package and reload parity |
| Explained but not fully implemented | larger training profiles; multi-device distribution; activation-memory modeling; cross-hardware reproducibility |
| Named and explicitly out of scope | pretrained-weight loading; SFT; RLHF/DPO; RAG; assistant behavior; tool use; quantized inference; serving; broad knowledge, semantic, reasoning, or capability claims |

### Key Takeaways

- Verify artifact identity before model construction, not after a suspicious run.
- One optimizer update is not one microbatch when gradients accumulate.
- A faithful resume needs optimizer, scheduler, counters, data identity, and every RNG stream.
- Select construction checkpoints with held-out validation, never with attractive prompt text.
- A base checkpoint is a reproducible next-token model artifact, not an assistant.